# 02 - Feature Engineering

This notebook demonstrates the feature engineering pipeline for churn prediction:

1. **Churn Definition**: Define and label churned vs retained customers
2. **RFM Features**: Recency, Frequency, Monetary metrics
3. **Engagement Features**: Conversion ratios and funnel metrics
4. **Trend Features**: Activity changes over time
5. **Feature Analysis**: Correlation and distribution analysis

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.churn_definition import ChurnLabeler, Windows
from src.features import FeatureEngineer

# Settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load events
loader = DataLoader(data_dir='../data')
events = loader.load_events()
print(f"Loaded {len(events):,} events")
print(f"Date range: {events['timestamp'].min().date()} to {events['timestamp'].max().date()}")

## 2. Define Churn

We define churn using a windowed approach:
- **Observation Window**: 60 days - period to build features from
- **Gap Period**: 7 days - buffer to prevent data leakage
- **Churn Window**: 30 days - if no purchase in this period, customer is churned

In [ ]:
# Configure churn windows
windows = Windows(
    obs=60,
    gap=7,
    chk=30
)
print(windows)

In [ ]:
# Initialize labeler and create labels
labeler = ChurnLabeler(windows=windows)

# Explain the business logic
print(labeler.explain())

In [ ]:
# Label customers
labels = labeler.label(
    events,
    min_txns=1  # Only include customers who purchased
)
print(f"\nLabeled {len(labels):,} customers")
labels.head()

In [ ]:
# Churn distribution
churn_dist = labels['churned'].value_counts()
print("Churn Distribution:")
print(churn_dist)
print(f"\nChurn Rate: {labels['churned'].mean():.1%}")

In [ ]:
# Visualize churn distribution
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2ecc71', '#e74c3c']
labels['churned'].value_counts().plot(kind='bar', ax=ax, color=colors)
ax.set_xticklabels(['Retained', 'Churned'], rotation=0)
ax.set_ylabel('Number of Customers')
ax.set_title('Churn Distribution')

# Add percentage labels
for i, (idx, val) in enumerate(labels['churned'].value_counts().items()):
    pct = val / len(labels) * 100
    ax.text(i, val + 50, f'{pct:.1f}%', ha='center')

plt.tight_layout()
plt.savefig('../figures/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Build Features

In [ ]:
# Get observation window events
obs_events = labeler.obs_events(events, labels)
print(f"Events in observation window: {len(obs_events):,}")

In [ ]:
# Initialize feature engineer and build features
engineer = FeatureEngineer(session_timeout_minutes=30)

features = engineer.build_features(
    events=obs_events,
    labels=labels
)

print(f"\nBuilt {len(features.columns) - 1} features for {len(features):,} customers")
features.head()

In [ ]:
# Feature descriptions
descriptions = engineer.get_feature_descriptions()
print("Feature Descriptions:")
print("-" * 60)
for feat, desc in descriptions.items():
    if feat in features.columns:
        print(f"{feat}:\n  {desc}\n")

In [ ]:
# Merge features with labels
df = features.merge(labels[['visitorid', 'churned']], on='visitorid')
print(f"Final dataset: {len(df):,} rows, {len(df.columns)} columns")

## 4. Feature Analysis

In [ ]:
# Feature statistics
feature_cols = [c for c in df.columns if c not in ['visitorid', 'churned']]
df[feature_cols].describe().T

In [ ]:
# Compare feature means by churn status
churn_comparison = df.groupby('churned')[feature_cols].mean().T
churn_comparison.columns = ['Retained', 'Churned']
churn_comparison['Difference'] = churn_comparison['Churned'] - churn_comparison['Retained']
churn_comparison['Pct_Diff'] = (churn_comparison['Difference'] / churn_comparison['Retained'] * 100).round(1)

print("Feature Comparison (Churned vs Retained):")
churn_comparison.sort_values('Pct_Diff', ascending=False)

In [ ]:
# Top differentiating features
top_diff = churn_comparison.sort_values('Pct_Diff', key=abs, ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in top_diff['Pct_Diff']]
top_diff['Pct_Diff'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('% Difference (Churned - Retained)')
ax.set_title('Top Differentiating Features')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../figures/top_diff_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions by churn status
key_features = ['days_since_purchase', 'transaction_count', 'v2p_rate', 'activity_trend']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    if feat in df.columns:
        for churned, color, label in [(0, '#2ecc71', 'Retained'), (1, '#e74c3c', 'Churned')]:
            data = df[df['churned'] == churned][feat]
            # Clip outliers for visualization
            data_clipped = data.clip(data.quantile(0.01), data.quantile(0.99))
            ax.hist(data_clipped, bins=30, alpha=0.6, label=label, color=color, density=True)
        ax.set_xlabel(feat)
        ax.set_ylabel('Density')
        ax.legend()
        ax.set_title(f'Distribution of {feat}')

plt.tight_layout()
plt.savefig('../figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix
corr_matrix = df[feature_cols + ['churned']].corr()

# Correlations with churn
churn_corr = corr_matrix['churned'].drop('churned').sort_values(key=abs, ascending=False)
print("Feature Correlations with Churn:")
print(churn_corr.head(15))

In [ ]:
# Correlation heatmap (top features)
top_features = churn_corr.head(10).index.tolist() + ['churned']
corr_subset = df[top_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation Matrix (Top Features)')
plt.tight_layout()
plt.savefig('../figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save Processed Data

In [ ]:
# Save features for modeling
df.to_parquet('../data/features/churn_features.parquet', index=False)
print(f"Saved features to ../data/features/churn_features.parquet")
print(f"Shape: {df.shape}")

## 6. Key Insights

### Top Predictive Signals

1. **Recency metrics** (days since last activity) - strongest churn predictors
2. **Engagement ratios** - view-to-purchase conversion indicates commitment
3. **Activity trends** - declining activity is a leading indicator
4. **Transaction frequency** - more transactions = lower churn risk

### Business Implications

- Early intervention is key - recency is highly predictive
- Engagement drop-offs signal at-risk customers
- Heavy purchasers are more loyal (lower churn)
- Session patterns can identify disengagement